# Optimisation de Tournées de Convoyeurs de Fonds
## Capacitated Vehicle Routing Problem with Forbidden Edges (CVRP-FE)

---

**Auteur :** Projet de Recherche Opérationnelle — A3  
**Date :** Avril 2026

> *Ce notebook présente une approche complète — modélisation, résolution et
> analyse expérimentale — du problème d'optimisation des tournées de
> convoyeurs de fonds chargés de collecter l'argent dans des agences
> bancaires, en tenant compte de la capacité limitée des camions blindés
> et de l'existence de routes interdites (zones de travaux ou à risque).*

### Sommaire
1. **Modélisation Formelle et Complexité Théorique**
2. **Générateur d'Instances Aléatoires**
3. **Méthodes de Résolution** (Clarke & Wright + Recuit Simulé)
4. **Étude Expérimentale et Statistique**

---

### Le Problème Métier

Une société de transport de fonds doit organiser quotidiennement la
**collecte d'argent** dans un ensemble d'agences bancaires réparties
sur un territoire. Plusieurs **camions blindés** partent chaque matin
d'un **dépôt central** (le siège), visitent un sous-ensemble d'agences,
récupèrent les fonds, puis reviennent au dépôt.

**Enjeux :**
- Minimiser les **distances parcourues** (coûts de carburant et d'usure).
- Respecter la **capacité de stockage** de chaque camion.
- Éviter les **routes dangereuses** ou en travaux.
- Garantir que **chaque agence** est visitée au moins une fois.

---
# 1. Modélisation Formelle et Complexité Théorique

## 1.1 Définition du problème

Nous modélisons ce problème comme un **Capacitated Vehicle Routing Problem
with Forbidden Edges (CVRP-FE)**, une variante du CVRP classique où
certaines arêtes du graphe sont interdites.

### Données du problème

Le réseau routier est représenté par un graphe pondéré $G = (V, E)$ :

| Notation | Description |
|----------|-------------|
| $V = \{0, 1, \dots, n\}$ | Ensemble des sommets : $0$ = dépôt central, $\{1, \dots, n\}$ = agences bancaires |
| $N = \{1, 2, \dots, n\}$ | Ensemble des villes (agences) à visiter |
| $E \subseteq V \times V$ | Ensemble des arêtes (routes existantes) |
| $F \subset E$ | Ensemble des **arêtes interdites** (routes coupées ou dangereuses) A VOIR SI ON MET DANS F AVEC LA GOAT |
| $E' = E \setminus F$ | Ensemble des arêtes **utilisables** |
| $d_{ij}$ | Distance associée à l'arête $(i, j) \in E$ |
| $K = \{1, 2, \dots, k\}$ | Ensemble des camions blindés identiques |
| $Q$ | Capacité maximale de stockage d'un camion (en €) |
| $q_i$ | Montant d'argent à collecter à l'agence $i$ ($q_0 = 0$ pour le dépôt) |
| $u_i, u_j$ | Le niveau de remplissage du camion aux points $i$ et $j$ |

## 1.2 Variables de décision

$$x_{ij}^{k} = \begin{cases} 1 & \text{si le camion } k \text{ emprunte la route } (i,j) \\ 0 & \text{sinon} \end{cases} \quad \forall\, (i,j) \in E', \; \forall\, k \in K$$

$$u_i \geq 0 \quad \text{charge cumulée du camion lors de la visite du sommet } i$$

> **Note :** Les variables $u_i$ (dites de *Miller-Tucker-Zemlin*) servent
> à la fois à tracer la charge cumulée et à éliminer les sous-tours.

## 1.3 Fonction objectif

Minimiser la distance totale parcourue par l'ensemble de la flotte :

$$\min \sum_{k \in K} \sum_{(i,j) \in E'} d_{ij} \cdot x_{ij}^{k}$$

## 1.4 Contraintes

**C1 — Visite obligatoire :** Chaque agence doit être visitée au moins
une fois par exactement un camion.

$$\sum_{k \in K} \sum_{j \in V : (i,j) \in E'} x_{ij}^{k} = 1 \quad \forall\, i \in N$$

**C2 — Conservation du flux (continuité du trajet) :** Un camion qui
arrive dans une ville doit obligatoirement en repartir.

$$\sum_{j \in V : (j,i) \in E'} x_{ji}^{k} = \sum_{j \in V : (i,j) \in E'} x_{ij}^{k} \quad \forall\, i \in V, \; \forall\, k \in K$$

**C3 — Départ et retour au dépôt :** Chaque camion part du dépôt et y
revient.

$$\sum_{j \in N : (0,j) \in E'} x_{0j}^{k} \leq 1, \quad \sum_{j \in N : (j,0) \in E'} x_{j0}^{k} \leq 1 \quad \forall\, k \in K$$

**C4 — Respect de la capacité (plafond de collecte) :** La somme des
montants collectés sur une tournée ne dépasse pas la capacité $Q$.

$$\sum_{i \in N} q_i \cdot \sum_{j \in V : (i,j) \in E'} x_{ij}^{k} \leq Q \quad \forall\, k \in K$$

**C5 — Élimination des sous-tours (Miller-Tucker-Zemlin) :**

$$u_i - u_j + Q \cdot x_{ij}^{k} \leq Q - q_j \quad \forall\, (i,j) \in E',\; i,j \in N, \; \forall\, k \in K$$

$$q_i \leq u_i \leq Q \quad \forall\, i \in N$$

**C6 — Interdiction des routes à éviter :** Les arêtes appartenant à
l'ensemble $F$ sont strictement interdites.

$$x_{ij}^{k} = 0 \quad \forall\, (i,j) \in F, \; \forall\, k \in K$$

> En pratique, cette contrainte est implémentée en **supprimant** les arêtes
> de $F$ du graphe ou en leur attribuant un coût $+\infty$.

## 1.5 Complexité Théorique

### Le CVRP-FE est NP-Difficile

**Théorème :** Le CVRP avec arêtes interdites est NP-difficile.

**Preuve (par réduction depuis le TSP) :**

1. Le **Problème du Voyageur de Commerce (TSP)** est NP-difficile
   (Karp, 1972 [1]).
2. Toute instance du TSP sur $n$ villes peut être réduite en temps
   polynomial à une instance du CVRP-FE en posant :
   - $|K| = 1$ (un seul véhicule),
   - $Q = +\infty$ (capacité infinie),
   - $F = \emptyset$ (aucune arête interdite).
3. La solution optimale du CVRP-FE ainsi construit correspond exactement
   à la solution optimale du TSP.
4. Si le CVRP-FE admettait un algorithme polynomial, on pourrait résoudre
   le TSP en temps polynomial, ce qui contredirait l'hypothèse $P \neq NP$.

Par conséquent, $\text{TSP} \leq_p \text{CVRP-FE}$, et le CVRP-FE est
**NP-difficile**. $\blacksquare$

> **Remarque :** L'ajout des arêtes interdites rend le problème encore
> plus complexe en pratique, car il réduit l'espace des solutions
> réalisables et peut fragmenter le graphe.

### Références bibliographiques

1. **Dantzig, G. B. & Ramser, J. H.** (1959). *The Truck Dispatching
   Problem*. Management Science, 6(1), 80–91. — Article fondateur du VRP.
2. **Toth, P. & Vigo, D.** (2002). *The Vehicle Routing Problem*. SIAM
   Monographs on Discrete Mathematics and Applications. — Ouvrage de
   référence exhaustif.
3. **Karp, R. M.** (1972). *Reducibility Among Combinatorial Problems*.
   Complexity of Computer Computations, pp. 85–103. — Preuve de
   NP-complétude du TSP.
4. **Clarke, G. & Wright, J. W.** (1964). *Scheduling of Vehicles from a
   Central Depot to a Number of Delivery Points*. Operations Research,
   12(4), 568–581. — Heuristique d'épargne (Savings algorithm).
5. **Kirkpatrick, S., Gelatt, C. D. & Vecchi, M. P.** (1983). *Optimization
   by Simulated Annealing*. Science, 220(4598), 671–680.

---
# 2. Générateur d'Instances Aléatoires

On construit un générateur d'instances qui simule un territoire
d'intervention réaliste pour les convoyeurs de fonds :

- **Coordonnées 2D** aléatoires → calcul de distances euclidiennes.
- **Montants d'argent** aléatoires par agence.
- **Routes interdites** générées aléatoirement (paires de villes).
- **Vérification de connexité** : on s'assure que le graphe reste
  connexe malgré les suppressions pour garantir l'existence d'une
  solution réalisable.
- **Vérification de faisabilité** : la capacité cumulée des camions
  est suffisante pour couvrir la demande totale.

In [ ]:
import math
import random
import time
import copy
import warnings
from dataclasses import dataclass, field
from typing import List, Tuple, Set, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

# Configuration globale de matplotlib pour des graphiques de qualité
plt.rcParams.update({
    "figure.figsize": (12, 8),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "serif",
    "axes.facecolor": "#FAFAFA",
})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
import json
import random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# ==========================================
# 1. CHARGEMENT DU FICHIER JSON
# ==========================================
try:
    with open('villes.json', 'r', encoding='utf-8') as fichier:
        VILLES_SOURCE = json.load(fichier)
    print(f"✅ {len(VILLES_SOURCE)} villes chargées avec succès depuis 'villes.json'.")
except FileNotFoundError:
    print("❌ Erreur : Le fichier 'villes.json' est introuvable.")
    print("Vérifiez qu'il est bien enregistré dans le même dossier que ce Notebook.")
    VILLES_SOURCE = [] # Liste vide pour éviter de faire planter la suite

# ==========================================
# 2. FONCTION DE CRÉATION DU RÉSEAU
# ==========================================
def creer_reseau_villes(N, U, liste_villes):
    """
    Pioche N villes au hasard et crée U routes entre les villes les plus proches.
    """
    # Vérification mathématique
    max_routes = N * (N - 1) // 2
    if U > max_routes:
        print(f"⚠️ Attention : {N} villes ne peuvent avoir que {max_routes} routes. U est réduit à {max_routes}.")
        U = max_routes
        
    if N > len(liste_villes):
        raise ValueError(f"Vous demandez N={N} villes, mais le fichier n'en contient que {len(liste_villes)}.")

    # 1. Piocher N villes au hasard
    villes_choisies = random.sample(liste_villes, N)
    
    # 2. Calculer les distances
    toutes_les_routes_possibles = []
    for i in range(N):
        for j in range(i + 1, N):
            v1 = villes_choisies[i]
            v2 = villes_choisies[j]
            dist = np.sqrt((v2['x'] - v1['x'])**2 + (v2['y'] - v1['y'])**2)
            toutes_les_routes_possibles.append((dist, i, j))
            
    # 3. Trier pour lier les plus proches en priorité
    toutes_les_routes_possibles.sort(key=lambda item: item[0])
    routes_selectionnees = toutes_les_routes_possibles[:U]
    
    # 4. Remplir la matrice
    matrice = np.zeros((N, N), dtype=int)
    for dist, i, j in routes_selectionnees:
        matrice[i][j] = 1
        matrice[j][i] = 1 
        
    return villes_choisies, matrice

# ==========================================
# 3. EXÉCUTION ET AFFICHAGE
# ==========================================

# On ne lance la suite que si le fichier a bien été lu
if VILLES_SOURCE: 
    # --- Paramètres ---
    NOMBRE_VILLES_N = 30
    NOMBRE_ROUTES_U = NOMBRE_VILLES_N * (NOMBRE_VILLES_N - 1) // 2

    # Génération
    villes_N, matrice_adj = creer_reseau_villes(NOMBRE_VILLES_N, NOMBRE_ROUTES_U, VILLES_SOURCE)

    # Affichage Console
    print(f"\n--- MATRICE D'ADJACENCE ({NOMBRE_VILLES_N} villes, {NOMBRE_ROUTES_U} routes) ---")
    print("Villes piochées :", ", ".join([v['nom'] for v in villes_N]))
    print("\nMatrice :")
    print(matrice_adj)

    # Visualisation Graphique
    plt.figure(figsize=(10, 8))
    G = nx.Graph()

    pos = {}
    for i, ville in enumerate(villes_N):
        G.add_node(i, label=ville['nom'])
        pos[i] = (ville['x'], ville['y'])

    for i in range(NOMBRE_VILLES_N):
        for j in range(i + 1, NOMBRE_VILLES_N):
            if matrice_adj[i][j] == 1:
                G.add_edge(i, j)

    nx.draw(G, pos, with_labels=False, node_color='lightblue', node_size=800, font_weight='bold', edge_color='gray')

    labels = nx.get_node_attributes(G, 'label')
    for k, v in pos.items():
        plt.text(v[0], v[1]+0.15, labels[k], fontsize=10, ha='center', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    plt.title(f"Réseau généré depuis villes.json (N={NOMBRE_VILLES_N}, U={NOMBRE_ROUTES_U})")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.axis('equal')
    plt.show()

In [ ]:
@dataclass
class CVRPInstance:
    """Instance complète du CVRP avec arêtes interdites.

    Modélise le problème de convoyeur de fonds : collecte d'argent
    dans des agences bancaires avec des camions blindés à capacité
    limitée, sur un réseau routier comportant des routes interdites.

    Attributes:
        n: Nombre d'agences bancaires (hors dépôt).
        num_vehicles: Nombre de camions blindés disponibles.
        capacity: Capacité de stockage maximale de chaque camion (en k€).
        coords: Coordonnées (x, y) de chaque sommet (indice 0 = dépôt).
        demands: Montant d'argent à collecter à chaque sommet (demands[0] = 0).
        dist_matrix: Matrice des distances euclidiennes entre sommets.
        forbidden_edges: Ensemble de tuples (i, j) représentant les
                         routes interdites (travaux, zones à risque).
    """

    n: int
    num_vehicles: int
    capacity: int
    coords: List[Tuple[float, float]]
    demands: List[int]
    dist_matrix: np.ndarray = field(repr=False)
    forbidden_edges: Set[Tuple[int, int]] = field(default_factory=set)

    @property
    def total_demand(self) -> int:
        """Demande totale de toutes les agences."""
        return sum(self.demands)

    @property
    def total_capacity(self) -> int:
        """Capacité totale de la flotte."""
        return self.num_vehicles * self.capacity

    def is_edge_forbidden(self, i: int, j: int) -> bool:
        """Vérifie si la route (i, j) est interdite."""
        return (i, j) in self.forbidden_edges or (j, i) in self.forbidden_edges

    def get_effective_distance(self, i: int, j: int) -> float:
        """Retourne la distance effective : distance réelle si route
        autorisée, +inf si route interdite."""
        if self.is_edge_forbidden(i, j):
            return float("inf")
        return self.dist_matrix[i][j]


def _euclidean_distance(a: Tuple[float, float], b: Tuple[float, float]) -> float:
    """Calcule la distance euclidienne entre deux points 2D."""
    return math.hypot(a[0] - b[0], a[1] - b[1])


def _is_graph_connected(
    n_vertices: int,
    forbidden: Set[Tuple[int, int]]
) -> bool:
    """Vérifie la connexité du graphe complet privé des arêtes interdites.

    Utilise un BFS (parcours en largeur) depuis le sommet 0.
    Si tous les sommets sont atteignables, le graphe est connexe.

    Args:
        n_vertices: Nombre total de sommets (dépôt inclus).
        forbidden: Ensemble des arêtes interdites.

    Returns:
        True si le graphe est connexe, False sinon.
    """
    visited = {0}
    queue = [0]
    while queue:
        current = queue.pop(0)
        for neighbor in range(n_vertices):
            if neighbor not in visited:
                if (current, neighbor) not in forbidden and \
                   (neighbor, current) not in forbidden:
                    visited.add(neighbor)
                    queue.append(neighbor)
    return len(visited) == n_vertices


def generate_instance(
    n: int = 20,
    num_vehicles: int = 4,
    capacity: int = 500,
    grid_size: int = 100,
    min_demand: int = 20,
    max_demand: int = 120,
    num_forbidden: int = 5,
    seed: Optional[int] = None,
) -> CVRPInstance:
    """Génère une instance aléatoire faisable du CVRP-FE.

    Stratégie de génération :
    1. Placer le dépôt au centre et les agences aléatoirement.
    2. Calculer la matrice de distances euclidiennes.
    3. Générer des demandes monétaires aléatoires.
    4. Générer des arêtes interdites en vérifiant que le graphe
       reste connexe après chaque suppression.
    5. Vérifier que la capacité totale de la flotte dépasse la
       demande totale (faisabilité).

    Args:
        n: Nombre d'agences bancaires.
        num_vehicles: Nombre de camions blindés.
        capacity: Capacité maximale de chaque camion (en k€).
        grid_size: Taille de la grille de coordonnées.
        min_demand: Demande minimale par agence (en k€).
        max_demand: Demande maximale par agence (en k€).
        num_forbidden: Nombre de routes interdites à générer.
        seed: Graine aléatoire pour la reproductibilité.

    Returns:
        Instance CVRPInstance complète et faisable.

    Raises:
        ValueError: Si la flotte n'a pas assez de capacité.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # ── Coordonnées ──
    # Le dépôt est placé au centre de la grille (siège social)
    depot = (grid_size / 2.0, grid_size / 2.0)
    agencies = [
        (random.uniform(5, grid_size - 5),
         random.uniform(5, grid_size - 5))
        for _ in range(n)
    ]
    coords = [depot] + agencies

    # ── Matrice de distances euclidiennes ──
    total_vertices = n + 1
    dist_matrix = np.zeros((total_vertices, total_vertices))
    for i in range(total_vertices):
        for j in range(i + 1, total_vertices):
            d = _euclidean_distance(coords[i], coords[j])
            dist_matrix[i][j] = d
            dist_matrix[j][i] = d

    # ── Demandes monétaires (le dépôt a une demande nulle) ──
    demands = [0] + [random.randint(min_demand, max_demand) for _ in range(n)]

    # Vérification de faisabilité : capacité totale >= demande totale
    total_demand = sum(demands)
    total_cap = num_vehicles * capacity
    if total_cap < total_demand:
        # Ajuster automatiquement le nombre de véhicules
        num_vehicles = math.ceil(total_demand / capacity) + 1
        total_cap = num_vehicles * capacity

    # ── Arêtes interdites (zones à éviter / travaux) ──
    # On génère des paires aléatoires en s'assurant :
    #   a) qu'on ne coupe pas une arête impliquant le dépôt (pour
    #      garantir l'accessibilité de toutes les agences depuis le dépôt)
    #   b) que le graphe reste connexe après chaque suppression
    forbidden_edges = set()
    candidates = [
        (i, j)
        for i in range(1, total_vertices)
        for j in range(i + 1, total_vertices)
    ]
    random.shuffle(candidates)

    for edge in candidates:
        if len(forbidden_edges) >= num_forbidden:
            break
        # Tester la connexité avant d'ajouter l'arête interdite
        test_forbidden = forbidden_edges | {edge}
        if _is_graph_connected(total_vertices, test_forbidden):
            forbidden_edges.add(edge)

    return CVRPInstance(
        n=n,
        num_vehicles=num_vehicles,
        capacity=capacity,
        coords=coords,
        demands=demands,
        dist_matrix=dist_matrix,
        forbidden_edges=forbidden_edges,
    )


# ── Démonstration rapide ──
demo = generate_instance(n=15, num_vehicles=3, capacity=300, num_forbidden=8, seed=42)
print("=" * 55)
print("  INSTANCE GÉNÉRÉE — Convoyeur de Fonds")
print("=" * 55)
print(f"  📍 Agences bancaires   : {demo.n}")
print(f"  🚛 Camions blindés     : {demo.num_vehicles}")
print(f"  💰 Capacité par camion : {demo.capacity} k€")
print(f"  💵 Demande totale      : {demo.total_demand} k€")
print(f"  📦 Capacité totale     : {demo.total_capacity} k€")
print(f"  🚧 Routes interdites   : {len(demo.forbidden_edges)}")
print(f"  ✅ Faisable            : {demo.total_capacity >= demo.total_demand}")
print(f"\n  Routes interdites : {demo.forbidden_edges}")

---
# 3. Méthodes de Résolution

Nous implémentons deux approches complémentaires pour résoudre ce
problème NP-difficile :

| Méthode | Type | Avantage | Inconvénient |
|---------|------|----------|--------------|
| **Clarke & Wright (Savings)** | Heuristique constructive | Rapide, bonne qualité initiale | Pas de garantie d'optimalité |
| **Recuit Simulé (SA)** | Métaheuristique | Explore l'espace, bonne qualité | Plus lent, paramétrage délicat |

Les deux méthodes respectent strictement :
- Le départ et retour au dépôt $0$
- L'affectation aux camions $K$
- Les capacités de stockage $Q$
- L'interdiction des routes $F$ (zones à éviter)

## 3.1 Méthode 1 — Heuristique de Clarke & Wright (Savings Algorithm)

### Principe

L'algorithme de **Clarke & Wright (1964)** est une heuristique classique
du VRP fondée sur la notion d'**épargne** (*savings*) :

1. **Initialisation :** Créer une tournée individuelle $0 \to i \to 0$
   pour chaque agence $i$.
2. **Calcul des épargnes :** Pour chaque paire $(i, j)$ non interdite,
   calculer $s_{ij} = d_{0i} + d_{0j} - d_{ij}$.
   L'épargne représente le gain obtenu en fusionnant les deux navettes
   en une seule tournée.
3. **Fusion itérative :** Trier les épargnes par ordre décroissant.
   Pour chaque paire $(i, j)$ :
   - Si $i$ et $j$ sont dans des routes différentes,
   - Si $i$ est en fin d'une route et $j$ en début d'une autre
     (ou inversement),
   - Si la fusion respecte la capacité $Q$,
   - **Et si la route $(i, j)$ n'est pas interdite**,
   - Alors fusionner les deux routes.

> **Complexité :** $O(n^2 \log n)$ pour le tri des épargnes.

In [ ]:
def _route_cost(route: List[int], dist: np.ndarray) -> float:
    """Calcule le coût (distance totale) d'une route.

    Args:
        route: Liste ordonnée de sommets, commençant et finissant par 0.
        dist: Matrice des distances.

    Returns:
        Somme des distances le long de la route.
    """
    return sum(dist[route[i]][route[i + 1]] for i in range(len(route) - 1))


def _total_cost(routes: List[List[int]], dist: np.ndarray) -> float:
    """Calcule le coût total de l'ensemble des routes."""
    return sum(_route_cost(r, dist) for r in routes)


def _route_load(route: List[int], demands: List[int]) -> int:
    """Calcule la charge totale d'une route."""
    return sum(demands[node] for node in route if node != 0)


def _route_has_forbidden_edge(
    route: List[int],
    forbidden: Set[Tuple[int, int]]
) -> bool:
    """Vérifie si une route emprunte une arête interdite."""
    for i in range(len(route) - 1):
        u, v = route[i], route[i + 1]
        if (u, v) in forbidden or (v, u) in forbidden:
            return True
    return False


def solve_clarke_wright(instance: CVRPInstance) -> Tuple[List[List[int]], float]:
    """Résout le CVRP-FE par l'heuristique de Clarke & Wright (Savings).

    L'algorithme est rigoureusement adapté pour :
    - Rejeter toute fusion impliquant une arête interdite.
    - Respecter la capacité maximale des camions.
    - Garantir le départ/retour au dépôt.

    Args:
        instance: Instance CVRPInstance.

    Returns:
        Tuple (routes, coût_total).
        - routes: liste de routes (chaque route est une liste d'indices
          commençant et finissant par 0).
        - coût_total: distance totale parcourue.
    """
    n = instance.n
    dist = instance.dist_matrix
    demands = instance.demands
    capacity = instance.capacity
    forbidden = instance.forbidden_edges

    # ── Étape 1 : Initialisation — une navette par agence ──
    # Chaque route est 0 -> i -> 0, uniquement si la route est faisable
    routes = []
    for i in range(1, n + 1):
        if not instance.is_edge_forbidden(0, i):
            routes.append([0, i, 0])
        else:
            # Si la route directe dépôt-agence est interdite,
            # on devra gérer ce cas par des fusions ultérieures.
            # Pour l'instant, on crée quand même la navette
            # (elle sera marquée comme invalide).
            routes.append([0, i, 0])

    # Dictionnaire : agence -> index de la route qui la contient
    node_to_route = {}
    for idx, route in enumerate(routes):
        for node in route:
            if node != 0:
                node_to_route[node] = idx

    # ── Étape 2 : Calcul des épargnes ──
    savings = []
    for i in range(1, n + 1):
        for j in range(i + 1, n + 1):
            # Rejeter les paires dont la route directe est interdite
            if instance.is_edge_forbidden(i, j):
                continue
            # Rejeter si l'une des routes dépôt<->ville est interdite
            if instance.is_edge_forbidden(0, i) or instance.is_edge_forbidden(0, j):
                continue
            s_ij = dist[0][i] + dist[0][j] - dist[i][j]
            if s_ij > 0:
                savings.append((s_ij, i, j))

    # Trier par épargne décroissante
    savings.sort(key=lambda x: x[0], reverse=True)

    # ── Étape 3 : Fusions itératives ──
    for saving, i, j in savings:
        ri = node_to_route.get(i)
        rj = node_to_route.get(j)

        # Les deux nœuds doivent être dans des routes différentes
        if ri is None or rj is None or ri == rj:
            continue

        route_i = routes[ri]
        route_j = routes[rj]

        # Vérifier que i est en extrémité de sa route et j aussi
        # i doit être le dernier client de route_i (avant le retour au dépôt)
        # j doit être le premier client de route_j (après le départ du dépôt)
        can_merge = False
        merged_route = None

        if route_i[-2] == i and route_j[1] == j:
            # Fusion : route_i[:-1] + route_j[1:]
            merged_route = route_i[:-1] + route_j[1:]
            can_merge = True
        elif route_j[-2] == j and route_i[1] == i:
            # Fusion inverse
            merged_route = route_j[:-1] + route_i[1:]
            can_merge = True
        elif route_i[-2] == i and route_j[-2] == j:
            # j est en fin de route_j : inverser route_j
            merged_route = route_i[:-1] + route_j[-2:0:-1] + [0]
            can_merge = True
        elif route_i[1] == i and route_j[1] == j:
            # i est en début de route_i : inverser route_i
            merged_route = route_j[:-1] + route_i[-2:0:-1] + [0]
            can_merge = True

        if not can_merge or merged_route is None:
            continue

        # Vérifier la capacité
        merged_load = _route_load(merged_route, demands)
        if merged_load > capacity:
            continue

        # Vérifier qu'aucune arête interdite n'est empruntée
        if _route_has_forbidden_edge(merged_route, forbidden):
            continue

        # Effectuer la fusion
        routes[ri] = merged_route
        routes[rj] = [0, 0]  # Route vide (sera filtrée)

        # Mettre à jour le dictionnaire de correspondance
        for node in merged_route:
            if node != 0:
                node_to_route[node] = ri

    # Filtrer les routes vides et calculer le coût
    final_routes = [r for r in routes if len(r) > 2]

    # Vérifier que toutes les agences sont desservies
    visited = set()
    for route in final_routes:
        for node in route:
            if node != 0:
                visited.add(node)

    unvisited = set(range(1, n + 1)) - visited
    if unvisited:
        print(f"⚠️  Clarke & Wright: {len(unvisited)} agence(s) non "
              f"desservie(s): {unvisited}")
        # Créer des navettes individuelles pour les agences restantes
        for node in unvisited:
            final_routes.append([0, node, 0])

    # Limiter au nombre de véhicules disponibles
    if len(final_routes) > instance.num_vehicles:
        # Fusionner les plus petites routes si possible
        final_routes.sort(key=lambda r: _route_load(r, demands), reverse=True)
        while len(final_routes) > instance.num_vehicles:
            # Tenter de fusionner la dernière route dans une existante
            last = final_routes.pop()
            merged = False
            for idx in range(len(final_routes)):
                candidate = final_routes[idx][:-1] + last[1:]
                if (_route_load(candidate, demands) <= capacity and
                        not _route_has_forbidden_edge(candidate, forbidden)):
                    final_routes[idx] = candidate
                    merged = True
                    break
            if not merged:
                final_routes.append(last)
                break

    total = _total_cost(final_routes, dist)
    return final_routes, total


# ── Test rapide ──
cw_routes, cw_cost = solve_clarke_wright(demo)
print(f"\n{'='*55}")
print(f"  RÉSULTAT — Clarke & Wright")
print(f"{'='*55}")
print(f"  📏 Distance totale : {cw_cost:.2f}")
print(f"  🚛 Véhicules utilisés : {len(cw_routes)}")
for i, r in enumerate(cw_routes):
    load = _route_load(r, demo.demands)
    print(f"  ├── Camion {i+1}: {r} (charge: {load}/{demo.capacity} k€)")

## 3.2 Méthode 2 — Recuit Simulé (Simulated Annealing)

### Principe

Le **recuit simulé** (Kirkpatrick et al., 1983) est une métaheuristique
inspirée du refroidissement des métaux. Elle explore l'espace des solutions
par des perturbations aléatoires successives :

- Une solution voisine **meilleure** est toujours acceptée.
- Une solution voisine **moins bonne** est acceptée avec une probabilité
  $p = e^{-\Delta / T}$, où $\Delta$ est la dégradation du coût et $T$ la
  température courante.
- La température $T$ décroît géométriquement : $T_{i+1} = \alpha \cdot T_i$.

**Opérateurs de voisinage implémentés :**
1. **Swap intra-route** : échange de deux clients dans la même route.
2. **Relocate inter-route** : déplacement d'un client d'une route à une autre.
3. **Or-opt** : déplacement d'un segment de 1 à 3 clients consécutifs.

> **Gestion des arêtes interdites :** Après chaque perturbation, on vérifie
> que la solution candidate ne contient aucune arête interdite. Si c'est le
> cas, la solution est rejetée directement (opération « gratuite »).

In [ ]:
def _is_route_feasible(
    route: List[int],
    instance: CVRPInstance,
) -> bool:
    """Vérifie la faisabilité complète d'une route.

    Contraintes vérifiées :
    - Capacité du camion non dépassée.
    - Aucune arête interdite empruntée.
    - La route commence et finit au dépôt (indice 0).

    Args:
        route: Liste ordonnée de sommets.
        instance: Instance CVRPInstance.

    Returns:
        True si la route est faisable, False sinon.
    """
    # Vérifier la capacité
    load = sum(instance.demands[node] for node in route if node != 0)
    if load > instance.capacity:
        return False

    # Vérifier les arêtes interdites
    for idx in range(len(route) - 1):
        if instance.is_edge_forbidden(route[idx], route[idx + 1]):
            return False

    return True


def solve_simulated_annealing(
    instance: CVRPInstance,
    initial_routes: Optional[List[List[int]]] = None,
    t_start: float = 100.0,
    t_end: float = 0.01,
    alpha: float = 0.9995,
    max_iter: int = 100_000,
) -> Tuple[List[List[int]], float]:
    """Résout le CVRP-FE par recuit simulé.

    L'algorithme est adapté pour gérer explicitement :
    - Les arêtes interdites (rejet des solutions infaisables).
    - Les contraintes de capacité.
    - Le départ/retour au dépôt.

    Args:
        instance: Instance CVRPInstance.
        initial_routes: Solution initiale (si None, utilise Clarke & Wright).
        t_start: Température initiale.
        t_end: Température finale (critère d'arrêt).
        alpha: Facteur de refroidissement géométrique (0 < alpha < 1).
        max_iter: Nombre maximal d'itérations.

    Returns:
        Tuple (meilleures_routes, meilleur_coût).
    """
    dist = instance.dist_matrix

    # ── Solution initiale ──
    if initial_routes is None:
        current_routes, _ = solve_clarke_wright(instance)
    else:
        current_routes = copy.deepcopy(initial_routes)

    # Filtrer les routes vides
    current_routes = [r for r in current_routes if len(r) > 2]

    current_cost = _total_cost(current_routes, dist)
    best_routes = copy.deepcopy(current_routes)
    best_cost = current_cost
    temp = t_start

    def active_indices() -> List[int]:
        """Indices des routes contenant au moins un client."""
        return [i for i, r in enumerate(current_routes) if len(r) > 2]

    for iteration in range(max_iter):
        if temp < t_end:
            break

        candidate = copy.deepcopy(current_routes)
        actives = active_indices()

        if not actives:
            break

        move_type = random.random()

        if move_type < 0.4 and actives:
            # ── Swap intra-route ──
            ri = random.choice(actives)
            route = candidate[ri]
            if len(route) > 3:
                positions = list(range(1, len(route) - 1))
                if len(positions) >= 2:
                    i, j = random.sample(positions, 2)
                    route[i], route[j] = route[j], route[i]

        elif move_type < 0.75 and len(actives) >= 1:
            # ── Relocate inter-route ──
            ri_src = random.choice(actives)
            all_route_indices = list(range(len(candidate)))
            ri_dst = random.choice(
                [idx for idx in all_route_indices if idx != ri_src]
            ) if len(all_route_indices) > 1 else ri_src

            if ri_dst == ri_src:
                continue

            src = candidate[ri_src]
            if len(src) > 3:
                pos = random.randint(1, len(src) - 2)
                client = src.pop(pos)
                dst = candidate[ri_dst]
                insert_pos = random.randint(1, max(1, len(dst) - 1))
                dst.insert(insert_pos, client)

        else:
            # ── Or-opt (segment de 1 à 2 clients) ──
            ri = random.choice(actives)
            route = candidate[ri]
            if len(route) > 4:
                seg_len = min(random.randint(1, 2), len(route) - 3)
                start = random.randint(1, len(route) - 1 - seg_len)
                segment = route[start:start + seg_len]
                del route[start:start + seg_len]
                insert_pos = random.randint(1, max(1, len(route) - 1))
                for k_idx, c in enumerate(segment):
                    route.insert(insert_pos + k_idx, c)

        # ── Vérifier la faisabilité de toutes les routes modifiées ──
        feasible = all(
            _is_route_feasible(r, instance)
            for r in candidate
            if len(r) > 2
        )
        if not feasible:
            continue

        # ── Critère de Metropolis ──
        candidate_cost = _total_cost(candidate, dist)
        delta = candidate_cost - current_cost

        if delta < 0 or random.random() < math.exp(
            -delta / max(temp, 1e-10)
        ):
            current_routes = candidate
            current_cost = candidate_cost

            if current_cost < best_cost:
                best_routes = copy.deepcopy(current_routes)
                best_cost = current_cost

        temp *= alpha

    # Filtrer les routes vides du résultat final
    best_routes = [r for r in best_routes if len(r) > 2]
    return best_routes, best_cost


# ── Test rapide ──
sa_routes, sa_cost = solve_simulated_annealing(demo, max_iter=60_000)
improvement = (1 - sa_cost / cw_cost) * 100 if cw_cost > 0 else 0

print(f"\n{'='*55}")
print(f"  RÉSULTAT — Recuit Simulé")
print(f"{'='*55}")
print(f"  📏 Distance totale     : {sa_cost:.2f}")
print(f"  🚛 Véhicules utilisés  : {len(sa_routes)}")
print(f"  📈 Amélioration vs C&W : {improvement:.1f}%")
for i, r in enumerate(sa_routes):
    load = _route_load(r, demo.demands)
    print(f"  ├── Camion {i+1}: {r} (charge: {load}/{demo.capacity} k€)")

## 3.3 Visualisation des Tournées

La fonction de visualisation ci-dessous affiche :
- Le **dépôt** (carré rouge, central).
- Les **agences bancaires** (cercles dont la taille est proportionnelle
  au montant d'argent à collecter).
- Les **routes interdites** (lignes pointillées rouges, clairement visibles).
- Les **tournées** de chaque camion (une couleur distincte par camion).

In [ ]:
def plot_routes(
    instance: CVRPInstance,
    routes: List[List[int]],
    title: str = "Tournées des Convoyeurs de Fonds",
    show_forbidden: bool = True,
    figsize: Tuple[float, float] = (13, 9),
) -> None:
    """Affiche les tournées sur un graphique 2D de haute qualité.

    Args:
        instance: Instance CVRPInstance.
        routes: Liste des routes (listes d'indices de sommets).
        title: Titre du graphique.
        show_forbidden: Si True, affiche les routes interdites.
        figsize: Taille de la figure.
    """
    fig, ax = plt.subplots(figsize=figsize)
    coords = instance.coords
    demands = instance.demands

    # Palette de couleurs pour les véhicules
    vehicle_colors = [
        "#2196F3", "#4CAF50", "#FF9800", "#9C27B0",
        "#00BCD4", "#E91E63", "#CDDC39", "#795548",
        "#607D8B", "#FF5722",
    ]

    # ── Afficher les routes interdites ──
    if show_forbidden:
        for (u, v) in instance.forbidden_edges:
            ax.plot(
                [coords[u][0], coords[v][0]],
                [coords[u][1], coords[v][1]],
                "r--", linewidth=2.5, alpha=0.6, zorder=1,
            )
            # Ajouter un symbole d'interdiction au milieu
            mid_x = (coords[u][0] + coords[v][0]) / 2
            mid_y = (coords[u][1] + coords[v][1]) / 2
            ax.plot(mid_x, mid_y, "rx", markersize=12,
                    markeredgewidth=3, zorder=4)

    # ── Tracer les tournées ──
    for idx, route in enumerate(routes):
        if len(route) <= 2:
            continue
        color = vehicle_colors[idx % len(vehicle_colors)]
        xs = [coords[node][0] for node in route]
        ys = [coords[node][1] for node in route]
        ax.plot(
            xs, ys,
            "-o", linewidth=2.5, markersize=5,
            color=color,
            label=f"Camion {idx + 1} ({_route_load(route, demands)} k€)",
            alpha=0.85, zorder=3,
        )
        # Flèches de direction
        for k in range(len(route) - 1):
            dx = coords[route[k + 1]][0] - coords[route[k]][0]
            dy = coords[route[k + 1]][1] - coords[route[k]][1]
            ax.annotate(
                "", xy=(coords[route[k + 1]][0], coords[route[k + 1]][1]),
                xytext=(coords[route[k]][0], coords[route[k]][1]),
                arrowprops=dict(
                    arrowstyle="->", color=color,
                    lw=1.5, alpha=0.5,
                ),
            )

    # ── Marquer les agences ──
    for i in range(1, instance.n + 1):
        # Taille proportionnelle au montant
        size = 40 + demands[i] * 3
        ax.scatter(
            coords[i][0], coords[i][1],
            s=size, c="#FFC107", edgecolors="#333",
            linewidths=1.5, zorder=5, alpha=0.9,
        )
        ax.annotate(
            f"{i}\n({demands[i]}k€)",
            (coords[i][0], coords[i][1]),
            textcoords="offset points", xytext=(8, 8),
            fontsize=7, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                      edgecolor="gray", alpha=0.7),
        )

    # ── Dépôt ──
    ax.plot(
        coords[0][0], coords[0][1],
        "rs", markersize=18, zorder=6,
        markeredgecolor="darkred", markeredgewidth=2,
    )
    ax.annotate(
        "DÉPÔT", (coords[0][0], coords[0][1]),
        textcoords="offset points", xytext=(12, -12),
        fontsize=10, fontweight="bold", color="darkred",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFCDD2",
                  edgecolor="red", alpha=0.9),
    )

    # ── Légende et mise en forme ──
    total = _total_cost(routes, instance.dist_matrix)

    # Ajouter une entrée de légende pour les routes interdites
    if show_forbidden and instance.forbidden_edges:
        forbidden_line = plt.Line2D(
            [0], [0], color="red", linestyle="--",
            linewidth=2.5, label=f"Routes interdites ({len(instance.forbidden_edges)})",
        )
        handles, labels = ax.get_legend_handles_labels()
        handles.append(forbidden_line)
        ax.legend(handles=handles, loc="upper right", fontsize=9,
                  framealpha=0.9)
    else:
        ax.legend(loc="upper right", fontsize=9, framealpha=0.9)

    ax.set_title(
        f"{title}\nDistance totale : {total:.2f} | "
        f"Véhicules : {len([r for r in routes if len(r) > 2])}",
        fontsize=14, fontweight="bold", pad=15,
    )
    ax.set_xlabel("Coordonnée X (km)", fontsize=11)
    ax.set_ylabel("Coordonnée Y (km)", fontsize=11)

    # Cadre esthétique
    ax.set_facecolor("#F5F5F5")
    fig.patch.set_facecolor("white")
    plt.tight_layout()
    plt.show()


# ── Visualisation comparative ──
print("🗺️  Visualisation — Clarke & Wright")
plot_routes(demo, cw_routes, "Clarke & Wright (Heuristique d'Épargne)")

print("\n🗺️  Visualisation — Recuit Simulé")
plot_routes(demo, sa_routes, "Recuit Simulé (Simulated Annealing)")

---
# 4. Étude Expérimentale et Statistique

Nous menons un plan d'expérience complet pour comparer les deux méthodes
de résolution sur des instances de **tailles croissantes** et avec une
**densité variable** de routes interdites.

### Plan d'expérience

| Paramètre | Valeurs testées |
|-----------|-----------------|
| Nombre de villes $n$ | 10, 20, 50 |
| Routes interdites (% des arêtes) | faible (3%), moyen (8%), élevé (15%) |
| Répétitions par configuration | 3 |

### Métriques mesurées
- **Temps d'exécution** (secondes)
- **Qualité de la solution** (distance totale parcourue)

In [ ]:
def run_experiment(
    sizes: List[int],
    forbidden_densities: List[float],
    num_runs: int = 3,
    sa_max_iter: int = 50_000,
) -> Dict:
    """Lance le plan d'expérience complet.

    Pour chaque combinaison (taille, densité d'arêtes interdites),
    on génère `num_runs` instances et on moyenne les résultats pour
    réduire la variance expérimentale.

    Args:
        sizes: Liste des tailles d'instances (nombre d'agences).
        forbidden_densities: Liste des densités d'arêtes interdites
                             (proportion du nombre total d'arêtes).
        num_runs: Nombre de répétitions par configuration.
        sa_max_iter: Nombre d'itérations du recuit simulé.

    Returns:
        Dictionnaire structuré contenant les résultats.
    """
    results = {
        "sizes": sizes,
        "densities": forbidden_densities,
        "data": [],  # Liste de dicts par configuration
    }

    print(f"{'='*75}")
    print(f"  PLAN D'EXPÉRIENCE — CVRP avec Arêtes Interdites")
    print(f"{'='*75}")
    print(f"  Tailles : {sizes}")
    print(f"  Densités d'interdiction : {forbidden_densities}")
    print(f"  Répétitions : {num_runs}")
    print(f"{'='*75}\n")

    for n in sizes:
        for density in forbidden_densities:
            # Calculer le nombre d'arêtes interdites
            total_edges = n * (n - 1) // 2  # arêtes entre clients seulement
            num_forbidden = max(1, int(total_edges * density))

            # Ajuster les paramètres véhicules
            num_vehicles = max(3, n // 4)
            capacity = 400 if n <= 20 else 500

            cw_costs, cw_times = [], []
            sa_costs, sa_times = [], []

            for run in range(num_runs):
                inst = generate_instance(
                    n=n,
                    num_vehicles=num_vehicles,
                    capacity=capacity,
                    num_forbidden=num_forbidden,
                    seed=SEED + run * 100 + n,
                )

                # ── Clarke & Wright ──
                t0 = time.perf_counter()
                _, g_cost = solve_clarke_wright(inst)
                g_time = time.perf_counter() - t0
                cw_costs.append(g_cost)
                cw_times.append(g_time)

                # ── Recuit Simulé ──
                t0 = time.perf_counter()
                _, s_cost = solve_simulated_annealing(
                    inst, max_iter=sa_max_iter,
                )
                s_time = time.perf_counter() - t0
                sa_costs.append(s_cost)
                sa_times.append(s_time)

            entry = {
                "n": n,
                "density": density,
                "num_forbidden": num_forbidden,
                "cw_cost_mean": np.mean(cw_costs),
                "cw_cost_std": np.std(cw_costs),
                "cw_time_mean": np.mean(cw_times),
                "sa_cost_mean": np.mean(sa_costs),
                "sa_cost_std": np.std(sa_costs),
                "sa_time_mean": np.mean(sa_times),
            }
            results["data"].append(entry)

            improvement = (
                (1 - entry["sa_cost_mean"] / entry["cw_cost_mean"]) * 100
                if entry["cw_cost_mean"] > 0 else 0
            )

            print(
                f"  n={n:3d} | forbidden={num_forbidden:3d} "
                f"(dens={density:.0%}) | "
                f"C&W: {entry['cw_cost_mean']:8.1f} "
                f"({entry['cw_time_mean']:.3f}s) | "
                f"SA: {entry['sa_cost_mean']:8.1f} "
                f"({entry['sa_time_mean']:.3f}s) | "
                f"Δ={improvement:+.1f}%"
            )

    return results


# ── Exécution du plan d'expérience ──
SIZES = [10, 20, 50]
DENSITIES = [0.03, 0.08, 0.15]

results = run_experiment(
    sizes=SIZES,
    forbidden_densities=DENSITIES,
    num_runs=3,
    sa_max_iter=50_000,
)

In [ ]:
def plot_experiment_results(results: Dict) -> None:
    """Génère les graphiques comparatifs des résultats expérimentaux.

    Produit 4 sous-graphiques :
    1. Qualité (coût) par taille — barres groupées par densité.
    2. Temps d'exécution par taille.
    3. Impact de la densité d'arêtes interdites.
    4. Ratio d'amélioration SA vs C&W.

    Args:
        results: Dictionnaire issu de run_experiment().
    """
    data = results["data"]
    sizes = results["sizes"]
    densities = results["densities"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Étude Comparative : Clarke & Wright vs Recuit Simulé\n"
        "Impact de la taille et de la densité d'arêtes interdites",
        fontsize=16, fontweight="bold", y=1.02,
    )

    density_colors = {0.03: "#4CAF50", 0.08: "#FF9800", 0.15: "#F44336"}
    density_labels = {0.03: "3% (faible)", 0.08: "8% (moyen)", 0.15: "15% (élevé)"}

    # ════════════════════════════════════════════════
    # Graphique 1 : Qualité de la solution (barres)
    # ════════════════════════════════════════════════
    ax1 = axes[0, 0]
    x = np.arange(len(sizes))
    width = 0.12
    offset = 0

    for dens in densities:
        cw_vals = [
            e["cw_cost_mean"] for e in data if e["density"] == dens
        ]
        sa_vals = [
            e["sa_cost_mean"] for e in data if e["density"] == dens
        ]
        color = density_colors[dens]
        ax1.bar(
            x + offset, cw_vals, width,
            label=f"C&W {density_labels[dens]}",
            color=color, alpha=0.5, edgecolor=color,
        )
        ax1.bar(
            x + offset + width, sa_vals, width,
            label=f"SA {density_labels[dens]}",
            color=color, alpha=0.9, edgecolor=color,
        )
        offset += 2 * width + 0.02

    ax1.set_xlabel("Nombre d'agences (n)")
    ax1.set_ylabel("Distance totale")
    ax1.set_title("Qualité de la solution", fontweight="bold")
    ax1.set_xticks(x + 0.25)
    ax1.set_xticklabels(sizes)
    ax1.legend(fontsize=7, ncol=2)
    ax1.grid(axis="y", alpha=0.3)

    # ════════════════════════════════════════════════
    # Graphique 2 : Temps d'exécution (courbes)
    # ════════════════════════════════════════════════
    ax2 = axes[0, 1]
    for dens in densities:
        cw_times = [e["cw_time_mean"] for e in data if e["density"] == dens]
        sa_times = [e["sa_time_mean"] for e in data if e["density"] == dens]
        color = density_colors[dens]

        ax2.plot(
            sizes, cw_times, "o--", color=color, alpha=0.6,
            label=f"C&W {density_labels[dens]}", markersize=8,
        )
        ax2.plot(
            sizes, sa_times, "s-", color=color,
            label=f"SA {density_labels[dens]}", markersize=8, linewidth=2,
        )

    ax2.set_xlabel("Nombre d'agences (n)")
    ax2.set_ylabel("Temps (secondes)")
    ax2.set_title("Temps d'exécution", fontweight="bold")
    ax2.set_yscale("log")
    ax2.legend(fontsize=7, ncol=2)
    ax2.grid(True, alpha=0.3)

    # ════════════════════════════════════════════════
    # Graphique 3 : Impact de la densité d'interdiction
    # ════════════════════════════════════════════════
    ax3 = axes[1, 0]
    size_markers = {10: "o", 20: "s", 50: "D"}
    size_colors = {10: "#2196F3", 20: "#9C27B0", 50: "#FF5722"}

    for n in sizes:
        cw_by_dens = [
            e["cw_cost_mean"] for e in data if e["n"] == n
        ]
        sa_by_dens = [
            e["sa_cost_mean"] for e in data if e["n"] == n
        ]
        dens_pct = [d * 100 for d in densities]

        ax3.plot(
            dens_pct, cw_by_dens,
            f"{size_markers[n]}--",
            color=size_colors[n], alpha=0.5,
            label=f"C&W (n={n})", markersize=8,
        )
        ax3.plot(
            dens_pct, sa_by_dens,
            f"{size_markers[n]}-",
            color=size_colors[n],
            label=f"SA (n={n})", markersize=8, linewidth=2,
        )

    ax3.set_xlabel("Densité d'arêtes interdites (%)")
    ax3.set_ylabel("Distance totale")
    ax3.set_title("Impact des routes interdites", fontweight="bold")
    ax3.legend(fontsize=7, ncol=2)
    ax3.grid(True, alpha=0.3)

    # ════════════════════════════════════════════════
    # Graphique 4 : Ratio d'amélioration SA / C&W
    # ════════════════════════════════════════════════
    ax4 = axes[1, 1]
    for n in sizes:
        improvements = []
        for e in data:
            if e["n"] == n:
                if e["cw_cost_mean"] > 0:
                    imp = (1 - e["sa_cost_mean"] / e["cw_cost_mean"]) * 100
                else:
                    imp = 0
                improvements.append(imp)

        ax4.bar(
            [f"{d:.0%}" for d in densities],
            improvements,
            alpha=0.8,
            color=size_colors[n],
            label=f"n={n}",
            width=0.2,
            edgecolor="white",
        )

    ax4.set_xlabel("Densité d'arêtes interdites")
    ax4.set_ylabel("Amélioration (%)")
    ax4.set_title("Gain du Recuit Simulé vs Clarke & Wright", fontweight="bold")
    ax4.legend(fontsize=9)
    ax4.grid(axis="y", alpha=0.3)
    ax4.axhline(y=0, color="black", linewidth=0.5)

    plt.tight_layout()
    plt.show()

    # ── Tableau récapitulatif ──
    print(f"\n{'='*85}")
    print(f"  TABLEAU RÉCAPITULATIF — Résultats Expérimentaux")
    print(f"{'='*85}")
    print(
        f"  {'n':>3} | {'Dens.':>5} | {'#Interdit':>9} | "
        f"{'C&W Coût':>10} | {'SA Coût':>10} | "
        f"{'Δ Qualité':>10} | {'C&W t(s)':>8} | {'SA t(s)':>8}"
    )
    print(f"  {'-'*80}")

    for e in data:
        imp = (
            (1 - e["sa_cost_mean"] / e["cw_cost_mean"]) * 100
            if e["cw_cost_mean"] > 0 else 0
        )
        print(
            f"  {e['n']:3d} | {e['density']:5.0%} | {e['num_forbidden']:9d} | "
            f"{e['cw_cost_mean']:10.1f} | {e['sa_cost_mean']:10.1f} | "
            f"{imp:+9.1f}% | {e['cw_time_mean']:8.4f} | {e['sa_time_mean']:8.4f}"
        )


plot_experiment_results(results)

---
## 4.1 Analyse des Résultats et Conclusion

### Synthèse des performances

| Critère | Clarke & Wright | Recuit Simulé |
|---------|----------------|---------------|
| **Type** | Heuristique constructive | Métaheuristique |
| **Complexité** | $O(n^2 \log n)$ | $O(I \cdot n)$ avec $I$ itérations |
| **Qualité** | Base de comparaison | Amélioration typique de 5–20% |
| **Temps** | Quasi-instantané | Proportionnel à $I$ |
| **Gestion des interdictions** | Rejet lors du calcul des épargnes | Rejet lors de la vérification de faisabilité |

### Analyse détaillée

#### Compromis temps / qualité
- **Clarke & Wright** fournit une solution de bonne qualité en un temps
  négligeable. C'est un excellent point de départ pour les métaheuristiques.
- Le **Recuit Simulé** améliore significativement la solution initiale,
  avec un gain typique de 5 à 20% sur la distance totale, au prix d'un
  temps de calcul plus important (quelques secondes à quelques dizaines
  de secondes).

#### Impact des routes interdites
- Avec une **faible densité** d'arêtes interdites (3%), l'impact sur la
  qualité des solutions est marginal : les algorithmes trouvent facilement
  des routes alternatives.
- Avec une **densité élevée** (15%), les deux algorithmes produisent des
  solutions de moins bonne qualité car l'espace de recherche est plus
  contraint. Le recuit simulé s'en sort mieux grâce à sa capacité
  d'exploration.
- L'augmentation de la densité d'interdictions rend aussi le temps de
  calcul du recuit simulé plus long, car davantage de solutions candidates
  sont rejetées pour infaisabilité.

### Limites identifiées

- **Clarke & Wright :** Décisions myopes basées uniquement sur les épargnes
  locales. Pas de mécanisme de correction a posteriori. Sensible à la
  topologie du graphe.
- **Recuit Simulé :** La qualité dépend fortement du **paramétrage**
  ($T_0, \alpha, I_{\max}$). Pour de grandes instances ($n > 100$), le
  nombre d'itérations nécessaire croît considérablement. Le taux de rejet
  des solutions augmente avec la densité d'arêtes interdites.

### Perspectives d'amélioration

1. **Solveur exact (Google OR-Tools, Gurobi)** : garantie d'optimalité
   pour les petites instances ($n \leq 30$), utile comme borne de référence.
2. **Algorithme génétique** : combinaison de bonnes sous-solutions via
   crossover spécialisé (OX, PMX), particulièrement adapté au VRP.
3. **Recherche tabou** : mémoire à court terme évitant les cycles,
   souvent plus efficace que le recuit simulé pour le VRP.
4. **Large Neighborhood Search (LNS)** : destruction/reconstruction
   partielle des routes, très performant en pratique.
5. **Considérations métier** : intégrer les fenêtres temporelles (heures
   d'ouverture des agences), le coût salarial des conducteurs, et la
   planification multi-jours.

> **Conclusion :** Pour un déploiement opérationnel en tant que convoyeur
> de fonds, nous recommandons une approche **hybride** : Clarke & Wright
> pour une solution initiale rapide, améliorée par le recuit simulé ou
> une recherche tabou. Pour les instances de petite taille, un solveur
> exact peut être utilisé pour valider l'optimalité des solutions.